# CNN baseline — U-Net + ResNet50 on Sen1Floods11 (issue #19)Run this on **Kaggle with a T4 GPU** (Settings → Accelerator → GPU T4 x2; one is enough).Budget is ~30 GPU-h/week with 12-hour sessions, which beats free Colab for this.Expected: ~31 iterations/epoch over the 252 hand-labelled training chips,roughly 1–2 min/epoch, so 100 epochs is about 2–3 hours.Issue #19 expects water IoU in the **0.55–0.65** range rather than the publishedfigure exactly. A shortfall is the expected reproduction gap — report it, don't hide it.

## 1. Get the code

In [ ]:
!git clone -q https://github.com/cs59300-deep-learning/nisar-flood-mapping.git%cd nisar-flood-mapping!pip install -q segmentation-models-pytorch timm rasterio pyyaml

## 2. Get the dataTwo options. The GCS download fetches 446 chips plus labels and takes a while;if someone has mirrored Sen1Floods11 as a Kaggle dataset, attach it in thesidebar instead and point `--root` at `/kaggle/input/<name>`.

In [ ]:
!PYTHONPATH=src python -m nisar_flood.cli download --root data/sen1floods11!PYTHONPATH=src python -m nisar_flood.cli check    --root data/sen1floods11

`check` should print 252 / 89 / 90 / 15 for train / val / test / bolivia,no missing files, and inputs inside [0, 1]. **Stop here if it does not.**

## 3. Train

In [ ]:
!PYTHONPATH=src python -m nisar_flood.cli train --config configs/unet_resnet50.yaml --seed 0

Writes to `runs/unet_resnet50/seed0/`: `history.csv`, `curves.png`, `best.pt`,`last.pt`, `summary.json`. Checkpoint selection uses the validation image-wisemean water IoU, matching the Sen1Floods11 definition.

## 4. Evaluate on the held-out splits

In [ ]:
!PYTHONPATH=src python -m nisar_flood.cli evaluate \    --checkpoint runs/unet_resnet50/seed0/best.pt

`test` is the headline number for #19. `bolivia` is Sen1Floods11's out-of-distributionsplit and is reported separately — it is the closest thing in the source domain to thetransfer question E4 asks about NISAR, so it is worth a line in the report.

## 5. Keep the results

In [ ]:
import shutil, pathlib# Kaggle only persists /kaggle/working. Copy the small artefacts out, not the checkpoints.dst = pathlib.Path("/kaggle/working/unet_resnet50_seed0"); dst.mkdir(parents=True, exist_ok=True)for name in ["history.csv", "curves.png", "summary.json", "eval.json"]:    src = pathlib.Path("runs/unet_resnet50/seed0") / name    if src.exists():        shutil.copy(src, dst / name)print(sorted(p.name for p in dst.iterdir()))

Then commit `curves.png`, `history.csv` and `eval.json` into `reports/` — those are#19's acceptance criteria. **Do not commit `best.pt`** (~130 MB): checkpoints go to theHugging Face dataset repo, per `docs/data_sources.md`.